# Sentiment Analysis

### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import spacy
from sklearn.model_selection import train_test_split
import re
from transformers import AutoTokenizer
from sklearn.preprocessing import LabelEncoder

/home/harsh/Work/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Importing Data & Splitting 

In [2]:
data = pd.read_csv('../Data/sentiment_analysis.csv', index_col= False)
nlp = spacy.load("en_core_web_sm")

y = data['sentiment']
X = data.drop(columns= ['sentiment'])

train, temp, y_train, y_temp = train_test_split(
    X, y, test_size= 0.3, random_state= 42, stratify= y
)

test, validation, y_test, y_validation = train_test_split(
    temp, y_temp, test_size= 0.5, random_state= 42, stratify= y_temp
)

### Pre-Processing Data

In [3]:
# lowercased the whole text
def lowercase(df):
    df = df.copy()
    df['text'] = df['text'].str.lower()
    return df

# removed Urls from the text
def remove_urls(df):
    df = df.copy()
    df['text'] = df['text'].apply(lambda x: re.sub(r'https\S+|www\S+', '', x))
    return df

# removed numbers from the text
def remove_numbers(df):
    df = df.copy()
    df['text'] = df['text'].apply(lambda x: re.sub(r'\d+', '', x))
    return df

# handling stopwords with removing negation terms
negations = {'not', 'no', 'nor', 'never', "n't", "cannot", "none", "neither"}
stopwords = nlp.Defaults.stop_words - negations

def spacy_clean(df):
    df = df.copy()
    df['text'] = df['text'].apply(
        lambda x: ' '. join([
            t.lemma_ for t in nlp(x)
            if not t.is_punct and t.text not in stopwords
        ])
    )
    return df

# the above function do the work of all three functions 
# def remove_stopwords(df):
#     df = df.copy()
#     df['text'] = df['text'].apply(lambda x: ' '.join([t.text for t in nlp(x) if t.text not in stopwords]))
#     return df
# # lemmatize 
# def lemmatize(df):
#     df = df.copy()
#     df['text'] = df['text'].apply(lambda x: ' '.join([t.lemma_ for t in nlp(x)]))
#     return df
# # punctuation removal 
# def remove_punctuation(df):
#     df = df.copy()
#     df['text'] = df['text'].apply(lambda x: ' '.join([t.text for t in nlp(x) if not t.is_punct]))
#     return df

# As I'm using bert model then I don't need these steps so I am just removing them from from clean text.

def clean_text(df):
    df = df.copy()
    df = lowercase(df)
    df = remove_urls(df)
    # df = remove_numbers(df)
    # df = spacy_clean(df)
    return df

train = clean_text(train)
test = clean_text(test)
validation = clean_text(validation)

### Tokenizer with BERT

In [4]:
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def token(df):
    return tokenizer(
        list(df['text']),
        padding = True,
        truncation = True,
        max_length = 256,
        return_tensors = 'pt'
    )

train = token(train)
test = token(test)
validation = token(validation)

### Label Encoding the output

In [5]:
encoder = LabelEncoder()

y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)
y_validation = encoder.transform(y_validation)